# Import Packages

We'll make use of the following packages:
- `numpy` is a package for scientific computing in python.
- `pandas` A powerful Python library for data manipulation and analysis.
- `seaborn` A data visualization library based on matplotlib.
- `scikit-learn` A comprehensive library for machine learning in Python.
- `kaggle` Using Kaggle API to download data.

In [67]:
import numpy as np
import pandas as pd
import kagglehub
import os
import seaborn as sns
import matplotlib.pyplot as plt
from collections import defaultdict


# Download Data

In [2]:
# Download latest version
path = kagglehub.dataset_download("taeefnajib/used-car-price-prediction-dataset")

print(os.listdir(path))

['used_cars.csv']


In [3]:
# Load the dataset
df = pd.read_csv(os.path.join(path, "used_cars.csv"))

# Data preprocessing

## Check data

In [4]:
df.head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,Ford,Utility Police Interceptor Base,2013,"51,000 mi.",E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$10,300"
1,Hyundai,Palisade SEL,2021,"34,742 mi.",Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,"$38,005"
2,Lexus,RX 350 RX 350,2022,"22,372 mi.",Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,"$54,598"
3,INFINITI,Q50 Hybrid Sport,2015,"88,900 mi.",Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,Yes,"$15,500"
4,Audi,Q3 45 S line Premium Plus,2021,"9,835 mi.",Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,"$34,999"


## 📝 Initial Data Glance – Observations & Next Steps

### Overview

- **Columns:**  
  The dataset contains **12 columns**: `brand`, `model`, `model_year`, `milage`, `fuel_type`, `engine`, `transmission`, `ext_col`, `int_col`, `accident`, `clean_title`, `price`.

- **Data Types:**  
  Most columns are **object type** (categorical or text). Data transformation will be required (label encoding, one-hot encoding, parsing text fields).

- **Brand & Model:**  
  - Some `model` values are duplicated or concatenated (e.g., `RX 350 RX 350`).
  - Will inspect and **clean/split/merge brand and model** to ensure unique, consistent values.

- **Feature Extraction:**  
  - Columns such as `engine` and `transmission` contain multiple details (e.g., horsepower, engine size, cylinder count, speed type) that can be **parsed into new features**.

- **Missing Values:**  
  - Nulls detected in several columns (e.g., `clean_title`).  
  - Will analyze missingness and apply appropriate imputation (mode, new category, or predictive imputation).

- **Target Variable:**  
  - The `price` column is a string with currency symbol and commas—needs to be cleaned and converted to numeric.

- **Formatting Issues:**  
  - Fields like `milage` and `price` contain units/symbols (e.g., "mi.", "$", ",")—will remove for numeric conversion.
  - `accident` and `clean_title` are categorical but may need binarization or mapping.

---

### Next Steps

- Clean and standardize all categorical and text fields.
- Parse and extract features from `engine` and `transmission` columns.
- Handle missing values with suitable imputation strategies.
- Convert `price` and `milage` to numeric types.
- Ensure brand/model consistency for analysis and modeling.

## Column Transformation

### Brand and Model

In [68]:
brand_list = sorted(df['brand'].unique())

by_letter = defaultdict(list)
for brand in brand_list:
    by_letter[brand[0].upper()].append(brand)
    

for letter in by_letter.keys():
    print(f"{letter}:{', '.join(by_letter[letter])}")

A:Acura, Alfa, Aston, Audi
B:BMW, Bentley, Bugatti, Buick
C:Cadillac, Chevrolet, Chrysler
D:Dodge
F:FIAT, Ferrari, Ford
G:GMC, Genesis
H:Honda, Hummer, Hyundai
I:INFINITI
J:Jaguar, Jeep
K:Karma, Kia
L:Lamborghini, Land, Lexus, Lincoln, Lotus, Lucid
M:MINI, Maserati, Maybach, Mazda, McLaren, Mercedes-Benz, Mercury, Mitsubishi
N:Nissan
P:Plymouth, Polestar, Pontiac, Porsche
R:RAM, Rivian, Rolls-Royce
S:Saab, Saturn, Scion, Subaru, Suzuki, smart
T:Tesla, Toyota
V:Volkswagen, Volvo


---
🏷️ Brand Name Inconsistencies

While reviewing the car brands, I noticed that several are missing their full names:

- **Alfa** → _Alfa Romeo_
- **Aston** → _Aston Martin_
- **Land** → _Land Rover_

Let’s take a closer look at these brands to ensure correct and consistent naming throughout the dataset.

In [21]:
df[df['brand'] == 'Alfa'].head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
151,Alfa,Romeo Stelvio Ti Sport,2020,"18,665 mi.",Gasoline,2.0L I4 16V GDI SOHC Turbo,8-Speed Automatic,Lunare White Metallic,Ice,None reported,Yes,"$35,645"
255,Alfa,Romeo Giulia Quadrifoglio,2022,"1,966 mi.",Gasoline,2.9L V6 24V GDI DOHC Twin Turbo,8-Speed Automatic,Verde,Black,None reported,NaN,"$75,900"
343,Alfa,Romeo Stelvio Ti,2020,"41,000 mi.",Gasoline,280.0HP 2.0L 4 Cylinder Engine Gasoline Fuel,8-Speed A/T,White,Black,None reported,Yes,"$32,400"
412,Alfa,Romeo Stelvio Quadrifoglio,2019,"26,500 mi.",Gasoline,505.0HP 2.9L V6 Cylinder Engine Gasoline Fuel,8-Speed A/T,Gray,Black,None reported,Yes,"$53,900"
414,Alfa,Romeo Stelvio Ti Sport,2020,"21,487 mi.",Gasoline,2.0L I4 16V GDI SOHC Turbo,8-Speed Automatic,Anodized Blue Metallic,Ice,None reported,Yes,"$35,345"


In [22]:
df[df['brand'] == 'Aston'].head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
11,Aston,Martin DBS Superleggera,2019,"22,770 mi.",Gasoline,715.0HP 5.2L 12 Cylinder Engine Gasoline Fuel,8-Speed A/T,Silver,Black,None reported,Yes,"$184,606"
93,Aston,Martin DBS Superleggera,2021,"2,165 mi.",Gasoline,5.2L V12 48V GDI DOHC Twin Turbo,8-Speed Automatic,Black,Black,None reported,Yes,"$279,950"
314,Aston,Martin DBX Base,2021,"2,353 mi.",Gasoline,4.0L V8 32V GDI DOHC Twin Turbo,9-Speed Automatic,Green,Sahara Tan,None reported,Yes,"$159,500"
535,Aston,Martin V8 Vantage Base,2008,"25,025 mi.",Gasoline,380.0HP 4.3L 8 Cylinder Engine Gasoline Fuel,6-Speed A/T,White,Black,At least 1 accident or damage reported,Yes,"$39,000"
610,Aston,Martin V8 Vantage Base,2008,"62,378 mi.",Gasoline,380.0HP 4.3L 8 Cylinder Engine Gasoline Fuel,M/T,Red,Beige,At least 1 accident or damage reported,Yes,"$33,995"


In [23]:
df[df['brand'] == 'Land'].head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
10,Land,Rover Range Rover Sport 3.0 Supercharged HST,2021,"27,608 mi.",Gasoline,V6,Automatic,Fuji White,Pimento / Ebony,None reported,NaN,"$73,897"
15,Land,Rover LR4 HSE,2013,"79,800 mi.",Gasoline,375.0HP 5.0L 8 Cylinder Engine Gasoline Fuel,A/T,White,Black,None reported,Yes,"$29,990"
80,Land,Rover Discovery Sport SE R-Dynamic,2020,"21,240 mi.",Gasoline,2.0 Liter,Automatic,White,Black,None reported,NaN,"$37,998"
110,Land,Rover LR4 HSE LUX Landmark Edition,2016,"144,000 mi.",Gasoline,340.0HP 3.0L V6 Cylinder Engine Gasoline Fuel,8-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$18,000"
120,Land,Rover Range Rover Sport 3.0L Supercharged HSE,2018,"104,700 mi.",Gasoline,V6,Automatic,Fuji White,Ivory / Ebony,At least 1 accident or damage reported,NaN,"$30,775"


---
### 🔧 Brand Name Corrections Needed

As predicted, these three brands require their brand and model names to be fixed for consistency:

- **Alfa** → _Alfa Romeo_
- **Aston** → _Aston Martin_
- **Land** → _Land Rover_

In [47]:
### Fixing Brand Name Inconsistencies

df_update = df.copy()

## Replace inconsistent brand names with full names
df_update['brand'] = df_update['brand'].replace({
    'Alfa': 'Alfa Romeo',
    'Aston': 'Aston Martin',
    'Land': 'Land Rover'
})

## Remove first word from model names for these brands
brand_name = ['Alfa Romeo', 'Aston Martin', 'Land Rover']

for brand in brand_name:

    df_update.loc[df_update['brand'] == brand, 'model'] = df_update.loc[df_update['brand'] == brand, 'model'].str.split().apply(lambda x: ' '.join(x[1:]) if isinstance(x, list) and len(x) > 1 else '')


In [48]:
df_update[df_update['brand'] == 'Alfa Romeo'].head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
151,Alfa Romeo,Stelvio Ti Sport,2020,"18,665 mi.",Gasoline,2.0L I4 16V GDI SOHC Turbo,8-Speed Automatic,Lunare White Metallic,Ice,None reported,Yes,"$35,645"
255,Alfa Romeo,Giulia Quadrifoglio,2022,"1,966 mi.",Gasoline,2.9L V6 24V GDI DOHC Twin Turbo,8-Speed Automatic,Verde,Black,None reported,NaN,"$75,900"
343,Alfa Romeo,Stelvio Ti,2020,"41,000 mi.",Gasoline,280.0HP 2.0L 4 Cylinder Engine Gasoline Fuel,8-Speed A/T,White,Black,None reported,Yes,"$32,400"
412,Alfa Romeo,Stelvio Quadrifoglio,2019,"26,500 mi.",Gasoline,505.0HP 2.9L V6 Cylinder Engine Gasoline Fuel,8-Speed A/T,Gray,Black,None reported,Yes,"$53,900"
414,Alfa Romeo,Stelvio Ti Sport,2020,"21,487 mi.",Gasoline,2.0L I4 16V GDI SOHC Turbo,8-Speed Automatic,Anodized Blue Metallic,Ice,None reported,Yes,"$35,345"


In [51]:
df_update[df_update['brand'] == 'Aston Martin'].head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
11,Aston Martin,DBS Superleggera,2019,"22,770 mi.",Gasoline,715.0HP 5.2L 12 Cylinder Engine Gasoline Fuel,8-Speed A/T,Silver,Black,None reported,Yes,"$184,606"
93,Aston Martin,DBS Superleggera,2021,"2,165 mi.",Gasoline,5.2L V12 48V GDI DOHC Twin Turbo,8-Speed Automatic,Black,Black,None reported,Yes,"$279,950"
314,Aston Martin,DBX Base,2021,"2,353 mi.",Gasoline,4.0L V8 32V GDI DOHC Twin Turbo,9-Speed Automatic,Green,Sahara Tan,None reported,Yes,"$159,500"
535,Aston Martin,V8 Vantage Base,2008,"25,025 mi.",Gasoline,380.0HP 4.3L 8 Cylinder Engine Gasoline Fuel,6-Speed A/T,White,Black,At least 1 accident or damage reported,Yes,"$39,000"
610,Aston Martin,V8 Vantage Base,2008,"62,378 mi.",Gasoline,380.0HP 4.3L 8 Cylinder Engine Gasoline Fuel,M/T,Red,Beige,At least 1 accident or damage reported,Yes,"$33,995"


In [53]:
df_update[df_update['brand'] == 'Land Rover'].head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
10,Land Rover,Range Rover Sport 3.0 Supercharged HST,2021,"27,608 mi.",Gasoline,V6,Automatic,Fuji White,Pimento / Ebony,None reported,NaN,"$73,897"
15,Land Rover,LR4 HSE,2013,"79,800 mi.",Gasoline,375.0HP 5.0L 8 Cylinder Engine Gasoline Fuel,A/T,White,Black,None reported,Yes,"$29,990"
80,Land Rover,Discovery Sport SE R-Dynamic,2020,"21,240 mi.",Gasoline,2.0 Liter,Automatic,White,Black,None reported,NaN,"$37,998"
110,Land Rover,LR4 HSE LUX Landmark Edition,2016,"144,000 mi.",Gasoline,340.0HP 3.0L V6 Cylinder Engine Gasoline Fuel,8-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$18,000"
120,Land Rover,Range Rover Sport 3.0L Supercharged HSE,2018,"104,700 mi.",Gasoline,V6,Automatic,Fuji White,Ivory / Ebony,At least 1 accident or damage reported,NaN,"$30,775"


---

Now that we’ve fixed the brand names, let’s address another issue: **duplicate model names**.  
For example, in row 2 we saw `Lexus RX 350 RX 350` as a model. These duplicates inflate the number of unique model groups.

**Next step:**  
We’ll clean the `model` column to remove repeated names and reduce redundancy in our model grouping.

### 🔧Clean Model names